In [13]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# 添加项目根目录到PATH
project_root = Path('../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

##### 初始化

In [14]:
from infrastructure.base_services.config_service import ConfigService
from infrastructure.base_services.cache_service import CacheService
from infrastructure.data_service.data_loading_service import DataLoadingService
from services.core_services.market_state_service import MarketStateService
from services.core_services.risk_assessment_service import RiskAssessmentService
from services.core_services.allocation_service import AllocationService
from services.core_services.sentiment_analysis_service import SentimentAnalysisService
from services.core_services.commodity_engine_service import CommodityEngineService
from services.core_services.macro_analysis_service import MacroAnalysisService
from services.core_services.option_pcr_service import OptionPCRService
from services.supplementary_services.cross_market_service import CrossMarketService
from services.supplementary_services.industry_rotation_service import IndustryRotationService
from services.supplementary_services.futures_analysis_service import FuturesAnalysisService
from services.visualization_service.visualization_service import VisualizationService
from services.threshold_service.threshold_service import ThresholdService  # ✅ V6.1新增
from utils.data_context_preparation import prepare_visualization_data_context

In [17]:
def verify_all_services():
    """验证所有服务初始化"""
    config = ConfigService('./system_config_v6.yaml')
    data_service = DataLoadingService(config)
    threshold_service = ThresholdService(config)
    
    services = {
        'RiskAssessmentService': RiskAssessmentService(data_service, config, threshold_service),
        'MarketStateService': MarketStateService(data_service, config, threshold_service),
        'AllocationService': AllocationService(config, threshold_service),
        'SentimentAnalysisService': SentimentAnalysisService(data_service, config, threshold_service),
        'CommodityEngineService': CommodityEngineService(data_service, config, threshold_service),
        'MacroAnalysisService': MacroAnalysisService(data_service, config, threshold_service),
        'OptionPCRService': OptionPCRService(data_service, config, threshold_service),
        'CrossMarketService': CrossMarketService(data_service, config, threshold_service),
        'IndustryRotationService': IndustryRotationService(data_service, config, threshold_service),
        'FuturesAnalysisService': FuturesAnalysisService(data_service, config, threshold_service)
    }
    
    print("\n✅ 所有服务初始化成功（无配置缺失警告）")
    print(f"   • 成功初始化 {len(services)} 个服务")
    print("   • 配置结构符合 V6.1 标准")
    print("   • ThresholdService 集成完成")

In [18]:
verify_all_services()

⚠️ 动态容忍度获取失败，回退静态配置: 'ThresholdService' object has 
⚠️ 配置路径缺失: adaptive_config → option_tolerance | 使用默认值 {}



✅ 所有服务初始化成功（无配置缺失警告）
   • 成功初始化 10 个服务
   • 配置结构符合 V6.1 标准
   • ThresholdService 集成完成


##### 服务实例化

In [19]:
config_service = ConfigService()
threshold_service = ThresholdService(config_service)
data_service = DataLoadingService(config_service)
risk_service = RiskAssessmentService(data_service, config_service, threshold_service)
allocation_service = AllocationService(config_service, threshold_service)
sentiment_service = SentimentAnalysisService(data_service, config_service,threshold_service)
commodity_service = CommodityEngineService(data_service, config_service,threshold_service)
macro_service = MacroAnalysisService(data_service, config_service,threshold_service)
option_pcr_service = OptionPCRService(data_service, config_service,threshold_service)   
cross_market_service = CrossMarketService(data_service, config_service,threshold_service)
industry_rotation_service = IndustryRotationService(data_service, config_service,threshold_service)
futures_analysis_service = FuturesAnalysisService(data_service, config_service,threshold_service)
visualization_service = VisualizationService()


⚠️ 动态容忍度获取失败，回退静态配置: 'ThresholdService' object has 
⚠️ 配置路径缺失: adaptive_config → option_tolerance | 使用默认值 {}


In [ ]:
from utils.config_utils import extract_and_validate_config, safe_config_get

In [ ]:
extract_and_validate_config(config_service=config_service, required_keys=[
    "market_state_thresholds",])    

In [ ]:

# 2. 验证动态阈值获取
dynamic_value = risk_service.threshold_service.get_threshold(
    'liquidity_warning_shrink',
    context={'volatility': 25.0}
)
assert isinstance(dynamic_value, float)
assert 0.5 <= dynamic_value <= 0.7  # 波动率高时阈值上移

# 3. 验证静态阈值回退
risk_service_no_threshold = RiskAssessmentService(data_service, config_service, threshold_service=None)
static_value = risk_service_no_threshold._get_stage_param('normal', 'exposure_cap', 0.20)
assert static_value == 0.20